# 💡 Notebook 5 — Insights & Recommendations
**Synthesising all analysis into actionable business recommendations**

Sections:
1. Executive Summary
2. Revenue Insights & Recommendations
3. Operational Insights & Recommendations (Cancellations, Refunds)
4. Customer Insights & Recommendations (Acquisition, Retention, RFM)
5. Restaurant & Cuisine Insights
6. Discount & Pricing Strategy
7. Prioritised Action Plan (Impact × Effort matrix)
8. KPI Targets & Success Metrics


In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (12, 5),
                     "axes.titlesize": 13, "axes.labelsize": 11})

BASE = r"C:\Users\rkuma\OneDrive\Desktop\Zomato"

def load(name):
    return pd.read_csv(os.path.join(BASE, f"Zomato  Order Data.xlsx - {name}.csv"))

customers   = load("Customer")
orders      = load("Orders")
restaurants = load("Restaurants")

orders["order_timestamp"] = pd.to_datetime(orders["order_timestamp"],
                                           format="%m/%d/%Y", errors="coerce")
orders["order_month"]   = orders["order_timestamp"].dt.to_period("M")
orders["order_quarter"] = orders["order_timestamp"].dt.to_period("Q")
orders["order_year"]    = orders["order_timestamp"].dt.year.astype("Int64")
orders["day_of_week"]   = orders["order_timestamp"].dt.day_name()
orders["hour"]          = orders["order_timestamp"].dt.hour
orders["discount_amount"] = orders["discount_amount"].fillna(0)
orders["delivery_fee"]    = orders["delivery_fee"].fillna(0)
orders["net_revenue"]     = orders["order_amount"] - orders["discount_amount"]
orders["is_discounted"]   = (orders["discount_amount"] > 0).astype(int)
orders["total_charge"]    = orders["net_revenue"] + orders["delivery_fee"]

customers["Signup_Time"]  = pd.to_datetime(customers["Signup_Time"],
                                           format="%d/%m/%Y", errors="coerce")
customers["signup_month"] = customers["Signup_Time"].dt.to_period("M")
customers["signup_year"]  = customers["Signup_Time"].dt.year.astype("Int64")

full = (orders
        .merge(restaurants, on="restaurant_id", how="left")
        .merge(customers,   left_on="customer_id",
               right_on="Customer_id", how="left"))

delivered  = full[full["order_status"] == "Delivered"].copy()
cancelled  = full[full["order_status"] == "Cancelled"].copy()
refunded   = full[full["order_status"] == "Refunded"].copy()

print(f"Orders: {len(orders):,} | Customers: {customers['Customer_id'].nunique():,} | Restaurants: {len(restaurants)}")
print(f"Date range: {orders['order_timestamp'].min().date()} to {orders['order_timestamp'].max().date()}")


In [ ]:

# ── Pre-compute all insight numbers ──────────────────────────
total_orders    = len(orders)
total_customers = customers["Customer_id"].nunique()
gmv             = orders["order_amount"].sum()
net_revenue     = delivered["net_revenue"].sum()
avg_order_value = delivered["order_amount"].mean()
delivery_rate   = (orders["order_status"]=="Delivered").mean()*100
cancel_rate     = (orders["order_status"]=="Cancelled").mean()*100
refund_rate     = (orders["order_status"]=="Refunded").mean()*100
total_discounts = orders["discount_amount"].sum()
avg_delivery_fee= orders["delivery_fee"].mean()

# City metrics
city_rev_s    = delivered.groupby("City")["net_revenue"].sum().sort_values(ascending=False)
city_cancel_s = (full.groupby("City")["order_status"]
                 .apply(lambda x: (x=="Cancelled").sum()/len(x)*100))
city_refund_s = (full.groupby("City")["order_status"]
                 .apply(lambda x: (x=="Refunded").sum()/len(x)*100))
city_aov_s    = delivered.groupby("City")["order_amount"].mean()

# Restaurant metrics
brand_rev_s    = delivered.groupby("restaurant_name")["net_revenue"].sum().sort_values(ascending=False)
brand_rating_s = restaurants.groupby("restaurant_name")["avg_rating"].mean().sort_values(ascending=False)
brand_refund_s = (full.groupby("restaurant_name")["order_status"]
                  .apply(lambda x: (x=="Refunded").sum()/len(x)*100))

# Cuisine
cuisine_rev_s = delivered.groupby("cuisine")["net_revenue"].sum().sort_values(ascending=False)

# Acquisition
acq_s = customers["Acquisition_channel"].value_counts()

# Discount
disc_orders   = orders[orders["discount_amount"]>0]["order_amount"]
nodisc_orders = orders[orders["discount_amount"]==0]["order_amount"]
disc_pct      = (orders["discount_amount"]>0).mean()*100

# RFM
snapshot_date = delivered["order_timestamp"].max() + pd.Timedelta(days=1)
rfm = delivered.groupby("customer_id").agg(
    recency   = ("order_timestamp", lambda x: (snapshot_date - x.max()).days),
    frequency = ("order_id","count"),
    monetary  = ("net_revenue","sum"),
).reset_index()
rfm["R"] = pd.qcut(rfm["recency"],  5, labels=[5,4,3,2,1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequency"].rank(method="first"),5,labels=[1,2,3,4,5]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"], 5, labels=[1,2,3,4,5]).astype(int)
def seg(row):
    r,f,m = row["R"],row["F"],row["M"]
    if r>=4 and f>=4 and m>=4: return "Champions"
    elif r>=3 and f>=3:         return "Loyal Customers"
    elif r>=4 and f<=2:         return "New Customers"
    elif r<=2 and f>=3:         return "At Risk"
    elif r<=2 and f<=2:         return "Lost Customers"
    else:                       return "Potential Loyalists"
rfm["Segment"] = rfm.apply(seg, axis=1)
champions   = rfm[rfm["Segment"]=="Champions"]
at_risk     = rfm[rfm["Segment"]=="At Risk"]
lost        = rfm[rfm["Segment"]=="Lost Customers"]

# Revenue lost
refund_rev  = full[full["order_status"]=="Refunded"]["order_amount"].sum()
cancel_rev  = full[full["order_status"]=="Cancelled"]["order_amount"].sum()

# MoM
mr_s = delivered.groupby("order_month")["net_revenue"].sum()
mom_growth = (mr_s.iloc[-1] - mr_s.iloc[-2]) / mr_s.iloc[-2] * 100 if len(mr_s)>=2 else 0

print("All metrics computed. Proceeding to insights...")


## 1. Executive Summary

In [ ]:

print("=" * 65)
print("  ZOMATO ANALYTICS — EXECUTIVE SUMMARY")
print("=" * 65)
print(f"""
SCALE
  Total Orders       : {total_orders:>10,}
  Unique Customers   : {total_customers:>10,}
  Restaurants        : {'200':>10}
  Cities             : {'8':>10}

REVENUE
  Gross Merch Value  : ₹{gmv/1e6:>9.2f}M
  Net Revenue        : ₹{net_revenue/1e6:>9.2f}M
  Avg Order Value    : ₹{avg_order_value:>9,.0f}
  Total Discounts    : ₹{total_discounts/1e6:>9.2f}M

OPERATIONS
  Delivery Rate      : {delivery_rate:>9.1f}%   (Target: 70%)
  Cancellation Rate  : {cancel_rate:>9.1f}%   (Target: <15%)
  Refund Rate        : {refund_rate:>9.1f}%   (Target: <10%)
  Revenue Lost (Cancel+Refund): ₹{(cancel_rev+refund_rev)/1e6:.2f}M

CUSTOMERS
  Champions (high-value)  : {len(champions):,} customers
  At Risk                 : {len(at_risk):,} customers
  Lost Customers          : {len(lost):,} customers
  MoM Revenue Growth      : {mom_growth:+.1f}%
""")
print("=" * 65)


## 2. Revenue Insights & Recommendations

In [ ]:

print("REVENUE INSIGHTS")
print("-" * 60)
print(f"1. Top revenue city: {city_rev_s.index[0]} (₹{city_rev_s.iloc[0]/1e6:.2f}M)")
print(f"   Bottom city: {city_rev_s.index[-1]} (₹{city_rev_s.iloc[-1]/1e6:.2f}M)")
print(f"   Revenue gap: {city_rev_s.iloc[0]/city_rev_s.iloc[-1]:.1f}x")
print()
print(f"2. Top cuisine: {cuisine_rev_s.index[0]} (₹{cuisine_rev_s.iloc[0]/1e6:.2f}M)")
print(f"   Bottom cuisine: {cuisine_rev_s.index[-1]} (₹{cuisine_rev_s.iloc[-1]/1e6:.2f}M)")
print()
print(f"3. Top brand: {brand_rev_s.index[0]} (₹{brand_rev_s.iloc[0]/1e6:.2f}M)")
print(f"4. Highest AOV city: {city_aov_s.idxmax()} (₹{city_aov_s.max():,.0f})")
print(f"   Lowest AOV city:  {city_aov_s.idxmin()} (₹{city_aov_s.min():,.0f})")
print(f"   AOV uplift opportunity: ₹{city_aov_s.max()-city_aov_s.min():,.0f} per order")
print()
print(f"5. MoM Revenue Growth: {mom_growth:+.1f}%")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Revenue by city with target line
city_rev_plot = city_rev_s.sort_values()
axes[0].barh(city_rev_plot.index, city_rev_plot.values/1e6,
             color=sns.color_palette("teal", len(city_rev_plot)))
axes[0].set_title("Net Revenue by City (₹M)", fontweight="bold")
axes[0].set_xlabel("Revenue (₹M)")
for i, v in enumerate(city_rev_plot.values/1e6):
    axes[0].text(v+0.02, i, f"₹{v:.2f}M", va="center", fontsize=9)

# AOV comparison
aov_sorted = city_aov_s.sort_values(ascending=False)
axes[1].bar(aov_sorted.index, aov_sorted.values,
            color=["#43AA8B" if v == aov_sorted.max() else
                   "#E63946" if v == aov_sorted.min() else "#264653"
                   for v in aov_sorted.values])
axes[1].axhline(aov_sorted.mean(), color="orange", linestyle="--",
                label=f"Average ₹{aov_sorted.mean():,.0f}")
axes[1].set_title("AOV by City — Opportunity Sizing", fontweight="bold")
axes[1].set_ylabel("Avg Order Value (₹)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()
for i, v in enumerate(aov_sorted.values):
    axes[1].text(i, v+2, f"₹{v:,.0f}", ha="center", fontsize=9)

plt.suptitle("Revenue Insights", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

print("""
REVENUE RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
R1. EXPAND TOP CITY MODEL
    Finding : {top_city} generates {top_city_rev} — highest in the network.
    Action  : Identify the top 3 operational practices (delivery speed,
              restaurant density, marketing mix) that make {top_city}
              successful and replicate them in the bottom 2 cities.
    Target  : Lift bottom 2 cities' revenue by 15% in 6 months.

R2. UPSELL IN LOW-AOV CITIES
    Finding : {low_aov_city} has ₹{low_aov:.0f} AOV vs ₹{high_aov:.0f} in {high_aov_city}.
    Action  : Add personalised "Frequently Added Together" upsell prompts
              at checkout. A/B test bundle deals (main + drink + side).
    Target  : +10% AOV in {low_aov_city} within 90 days.

R3. FOCUS ON TOP CUISINE
    Finding : {top_cuisine} generates {top_cuisine_rev:.1f}M vs {bot_cuisine} {bot_cuisine_rev:.1f}M.
    Action  : Feature {top_cuisine} restaurants prominently on the home
              screen. Use push notifications for {top_cuisine} offers.
    Target  : +20% orders in top cuisine segment.
""".format(
    top_city=city_rev_s.index[0], top_city_rev=f"₹{city_rev_s.iloc[0]/1e6:.2f}M",
    low_aov_city=city_aov_s.idxmin(), low_aov=city_aov_s.min(),
    high_aov=city_aov_s.max(), high_aov_city=city_aov_s.idxmax(),
    top_cuisine=cuisine_rev_s.index[0],
    top_cuisine_rev=cuisine_rev_s.iloc[0]/1e6,
    bot_cuisine=cuisine_rev_s.index[-1],
    bot_cuisine_rev=cuisine_rev_s.iloc[-1]/1e6,
))


## 3. Operational Insights & Recommendations

In [ ]:

print("OPERATIONAL INSIGHTS")
print("-" * 60)
print(f"Cancellation rate: {cancel_rate:.1f}% — ABOVE target of 15%")
print(f"Refund rate      : {refund_rate:.1f}% — ABOVE target of 10%")
print(f"Revenue at risk  : ₹{(cancel_rev+refund_rev)/1e6:.2f}M")
print()
print(f"Worst cancel city  : {city_cancel_s.idxmax()} ({city_cancel_s.max():.1f}%)")
print(f"Best cancel city   : {city_cancel_s.idxmin()} ({city_cancel_s.min():.1f}%)")
print()
print(f"Worst refund city  : {city_refund_s.idxmax()} ({city_refund_s.max():.1f}%)")
print(f"Worst refund brand : {brand_refund_s.idxmax()} ({brand_refund_s.max():.1f}%)")


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Cancel rate by city
cr = city_cancel_s.sort_values(ascending=True)
colors_c = ["#E63946" if v == cr.max() else "#264653" for v in cr.values]
axes[0,0].barh(cr.index, cr.values, color=colors_c)
axes[0,0].axvline(15, color="orange", linestyle="--", label="Target 15%")
axes[0,0].set_title("Cancel Rate by City vs Target (%)", fontweight="bold")
axes[0,0].set_xlabel("%"); axes[0,0].legend()
for i, v in enumerate(cr.values):
    axes[0,0].text(v+0.1, i, f"{v:.1f}%", va="center", fontsize=9)

# Refund rate by city
rr = city_refund_s.sort_values(ascending=True)
colors_r = ["#9C6B98" if v == rr.max() else "#264653" for v in rr.values]
axes[0,1].barh(rr.index, rr.values, color=colors_r)
axes[0,1].axvline(10, color="orange", linestyle="--", label="Target 10%")
axes[0,1].set_title("Refund Rate by City vs Target (%)", fontweight="bold")
axes[0,1].set_xlabel("%"); axes[0,1].legend()
for i, v in enumerate(rr.values):
    axes[0,1].text(v+0.1, i, f"{v:.1f}%", va="center", fontsize=9)

# Refund by brand (top 8)
rb = brand_refund_s.sort_values(ascending=False).head(8).sort_values()
axes[1,0].barh(rb.index, rb.values, color=sns.color_palette("Purples_r", len(rb)))
axes[1,0].axvline(10, color="orange", linestyle="--", label="Target 10%")
axes[1,0].set_title("Refund Rate by Brand (Top 8)", fontweight="bold")
axes[1,0].set_xlabel("%"); axes[1,0].legend()
for i, v in enumerate(rb.values):
    axes[1,0].text(v+0.1, i, f"{v:.1f}%", va="center", fontsize=9)

# Revenue impact of reducing cancel/refund to targets
current_lost   = (cancel_rev + refund_rev)
target_lost    = gmv * (0.15 + 0.10) / 1
revenue_rescue = current_lost * 0.30  # 30% reduction is realistic
axes[1,1].bar(["Current Lost Revenue","Rescue Potential (30% fix)"],
              [current_lost/1e6, revenue_rescue/1e6],
              color=["#E63946","#43AA8B"])
axes[1,1].set_title("Revenue Recovery Opportunity (₹M)", fontweight="bold")
axes[1,1].set_ylabel("₹ Millions")
for i, v in enumerate([current_lost/1e6, revenue_rescue/1e6]):
    axes[1,1].text(i, v+0.05, f"₹{v:.2f}M", ha="center", fontweight="bold", fontsize=11)

plt.suptitle("Operational Problem Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

print("""
OPERATIONAL RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
O1. REDUCE CANCELLATIONS — HIGH PRIORITY
    Finding : {wc_city} has {wc_rate:.1f}% cancel rate, {gap:.1f}pp above best city.
    Action  :
      • Deploy real-time ETA prediction at order placement.
      • Add "restaurant accepted" push notification within 2 min.
      • Offer 1-click reschedule instead of cancel for time-slot issues.
      • Investigate top 3 cancel reasons via in-app exit survey.
    Target  : Reduce network cancel rate from {cr:.1f}% to 15% in 6 months.
    Revenue : Recovering 30% of cancellations = ₹{rescue:.2f}M additional revenue.

O2. FIX REFUND LEAKAGE — HIGH PRIORITY
    Finding : {wb} has {wr:.1f}% refund rate. Refunds cost ₹{lost:.2f}M.
    Action  :
      • Enforce photo-verification of order before dispatch for brands
        with refund rate > 20%.
      • Set SLA: refund rate > 15% triggers mandatory quality audit.
      • Auto-flag repeat-refund customers and add verification layer.
    Target  : Reduce refund rate from {rr:.1f}% to 10% in 3 months.
""".format(
    wc_city=city_cancel_s.idxmax(), wc_rate=city_cancel_s.max(),
    gap=city_cancel_s.max()-city_cancel_s.min(),
    cr=cancel_rate,
    rescue=(cancel_rev+refund_rev)*0.30/1e6,
    wb=brand_refund_s.idxmax(), wr=brand_refund_s.max(),
    lost=refund_rev/1e6, rr=refund_rate,
))


## 4. Customer Insights & Recommendations

In [ ]:

seg_counts = rfm["Segment"].value_counts()
seg_rev    = rfm.groupby("Segment")["monetary"].sum().sort_values(ascending=False)

print("CUSTOMER SEGMENT PROFILE")
print("-" * 55)
for seg in seg_rev.index:
    n = seg_counts.get(seg, 0)
    r = seg_rev[seg]
    print(f"  {seg:<22} {n:>5,} customers  ₹{r/1e6:.2f}M revenue")

print(f"
Top 10% customers drive: "
      f"{rfm.nlargest(int(len(rfm)*0.1),'monetary')['monetary'].sum()/rfm['monetary'].sum()*100:.1f}%"
      f" of total revenue")
print(f"Acquisition top channel: {acq_s.index[0]} ({acq_s.iloc[0]:,} customers)")
print(f"Weakest channel        : {acq_s.index[-1]} ({acq_s.iloc[-1]:,} customers)")


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
palette = sns.color_palette("Set2", len(seg_counts))

# Segment pie
axes[0].pie(seg_counts.values, labels=seg_counts.index,
            autopct="%1.1f%%", colors=palette, startangle=140)
axes[0].set_title("Customer Mix by Segment")

# Revenue by segment
seg_rev_plot = seg_rev.sort_values()
axes[1].barh(seg_rev_plot.index, seg_rev_plot.values/1e6,
             color=palette[:len(seg_rev_plot)])
axes[1].set_title("Revenue by Segment (₹M)")
axes[1].set_xlabel("₹ Millions")
for i, v in enumerate(seg_rev_plot.values/1e6):
    axes[1].text(v+0.01, i, f"₹{v:.2f}M", va="center", fontsize=9)

# Acquisition channel
axes[2].bar(acq_s.index, acq_s.values,
            color=sns.color_palette("Set2", len(acq_s)))
axes[2].set_title("Customers by Acquisition Channel")
axes[2].set_ylabel("Customers")
axes[2].tick_params(axis="x", rotation=15)
for i, v in enumerate(acq_s.values):
    axes[2].text(i, v+5, str(v), ha="center", fontweight="bold", fontsize=10)

plt.suptitle("Customer Insights", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:

print("""
CUSTOMER RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
C1. PROTECT CHAMPIONS — HIGH PRIORITY
    Finding : {ch_n:,} Champions drive disproportionate revenue (₹{ch_rev:.2f}M).
    Action  :
      • VIP tier: free delivery, priority support, exclusive restaurant access.
      • Dedicated account manager for top 500 customers.
      • Early access to new restaurant launches.
    Target  : 0% churn among Champions in next quarter.

C2. WIN BACK AT-RISK & LOST CUSTOMERS
    Finding : {ar_n:,} At-Risk + {lost_n:,} Lost customers = ₹{lost_rev:.2f}M latent revenue.
    Action  :
      • 30-day win-back email + push sequence with escalating offers.
      • Personalised "We miss you" discount based on last order cuisine.
      • Survey churned customers for root-cause data.
    Target  : Re-activate 15% of At-Risk customers in 60 days.

C3. CONVERT POTENTIAL LOYALISTS
    Finding : {pl_n:,} Potential Loyalists need one more nudge.
    Action  :
      • Trigger a loyalty milestone reward after order #3 (e.g., ₹50 credit).
      • "Next order" push notification 3 days after delivery.
    Target  : Move 20% of Potential Loyalists to Loyal tier.

C4. DOUBLE DOWN ON TOP ACQUISITION CHANNEL
    Finding : {top_ch} brings {top_ch_n:,} customers — the most effective channel.
    Action  :
      • Increase budget allocation by 20% to this channel.
      • A/B test new creative formats.
      • Analyse cost-per-acquisition vs CLV to ensure profitability.
    Target  : +15% new customer growth from {top_ch} in Q next.
""".format(
    ch_n=len(champions), ch_rev=champions["monetary"].sum()/1e6,
    ar_n=len(at_risk), lost_n=len(lost),
    lost_rev=(at_risk["monetary"].sum()+lost["monetary"].sum())/1e6,
    pl_n=len(rfm[rfm["Segment"]=="Potential Loyalists"]),
    top_ch=acq_s.index[0], top_ch_n=acq_s.iloc[0],
))


## 5. Restaurant & Cuisine Insights

In [ ]:

low_rating   = brand_rating_s[brand_rating_s < 4.0]
high_refund  = brand_refund_s[brand_refund_s > 20]

print(f"Brands with avg rating < 4.0  : {len(low_rating)} brands")
print(f"  → {', '.join(low_rating.index.tolist())}")
print(f"
Brands with refund rate > 20%  : {len(high_refund)} brands")
print(f"  → {', '.join(high_refund.index.tolist()) if len(high_refund) else 'None'}")

print(f"
Top brand by revenue : {brand_rev_s.index[0]} (₹{brand_rev_s.iloc[0]/1e6:.2f}M)")
print(f"Top brand by rating  : {brand_rating_s.index[0]} ({brand_rating_s.iloc[0]:.2f}★)")


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Rating vs Revenue scatter
rest_scatter = (full.groupby("restaurant_name").agg(
    revenue=("net_revenue","sum"), orders=("order_id","count"),
    rating=("avg_rating","mean"), refund_rate=(
        "order_status", lambda x: (x=="Refunded").sum()/len(x)*100)
).reset_index())

sc = axes[0].scatter(rest_scatter["rating"], rest_scatter["revenue"]/1e6,
                     s=rest_scatter["orders"]/5, alpha=0.6,
                     c=rest_scatter["refund_rate"], cmap="RdYlGn_r",
                     edgecolors="grey", linewidths=0.5)
plt.colorbar(sc, ax=axes[0], label="Refund Rate %")
axes[0].axvline(4.0, color="red", linestyle="--", label="Rating threshold 4.0")
axes[0].set_xlabel("Avg Rating"); axes[0].set_ylabel("Revenue (₹M)")
axes[0].set_title("Rating × Revenue × Refund Rate
(bubble = order count)", fontweight="bold")
axes[0].legend()

# Top 10 revenue vs rating comparison
top10 = brand_rev_s.head(10).reset_index()
top10.columns = ["brand","revenue"]
top10 = top10.merge(brand_rating_s.reset_index().rename(
    columns={"restaurant_name":"brand","avg_rating":"rating"}), on="brand")
top10 = top10.sort_values("revenue", ascending=True)

ax2 = axes[1].twinx()
axes[1].barh(top10["brand"], top10["revenue"]/1e6, color="#264653", alpha=0.7, label="Revenue")
ax2.plot(top10["rating"], top10["brand"], "o--", color="#E63946", linewidth=2, label="Rating")
ax2.set_xlim(3, 5.5)
axes[1].set_xlabel("Revenue (₹M)")
ax2.set_xlabel("Avg Rating")
axes[1].set_title("Top 10 Brands: Revenue vs Rating", fontweight="bold")
axes[1].legend(loc="lower right"); ax2.legend(loc="upper left")

plt.suptitle("Restaurant & Cuisine Insights", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 6. Discount & Pricing Strategy

In [ ]:

from scipy.stats import ttest_ind

disc = orders[orders["discount_amount"]>0]["order_amount"]
nodisc = orders[orders["discount_amount"]==0]["order_amount"]
t, p = ttest_ind(disc, nodisc)

print("DISCOUNT ANALYSIS")
print("-" * 50)
print(f"Orders with discount   : {len(disc):,} ({len(disc)/total_orders*100:.1f}%)")
print(f"Orders without discount: {len(nodisc):,} ({len(nodisc)/total_orders*100:.1f}%)")
print(f"Discounted AOV         : ₹{disc.mean():,.0f}")
print(f"Non-discounted AOV     : ₹{nodisc.mean():,.0f}")
print(f"Lift                   : ₹{disc.mean()-nodisc.mean():,.0f} ({(disc.mean()-nodisc.mean())/nodisc.mean()*100:.1f}%)")
print(f"t-test p-value         : {p:.4f} ({'Significant' if p<0.05 else 'Not Significant'})")
print(f"Total discount spend   : ₹{total_discounts:,.0f}")
print(f"Discount as % of GMV   : {total_discounts/gmv*100:.2f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(disc,   bins=30, alpha=0.6, label=f"Discounted  μ=₹{disc.mean():,.0f}",   color="#2A9D8F")
axes[0].hist(nodisc, bins=30, alpha=0.6, label=f"No Discount μ=₹{nodisc.mean():,.0f}", color="#E63946")
axes[0].set_title("Order Amount Distribution: Discount vs No Discount")
axes[0].set_xlabel("Order Amount (₹)"); axes[0].legend()

discount_pct_dist = orders[orders["discount_amount"]>0]["discount_amount"] /                     orders[orders["discount_amount"]>0]["order_amount"] * 100
axes[1].hist(discount_pct_dist, bins=20, color="#E9C46A", edgecolor="white")
axes[1].axvline(discount_pct_dist.mean(), color="red", linestyle="--",
                label=f"Mean {discount_pct_dist.mean():.1f}%")
axes[1].set_title("Discount Depth Distribution (%)")
axes[1].set_xlabel("Discount as % of Order Amount"); axes[1].legend()

plt.suptitle("Discount Effectiveness", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


## 7. Prioritised Action Plan — Impact × Effort Matrix

In [ ]:

actions = [
    # (Label, Impact 1-10, Effort 1-10, Category)
    ("Reduce Cancellations
(ETA + notification)", 9, 5, "Operations"),
    ("Fix Refund Rate
(quality SLA)", 8, 4, "Operations"),
    ("Win-back At-Risk
Customers", 8, 3, "Customer"),
    ("Protect Champions
(VIP tier)", 9, 2, "Customer"),
    ("Low-AOV City
Upsell nudges", 6, 2, "Revenue"),
    ("Expand Top City
model", 7, 7, "Revenue"),
    ("Personalise
Discounts", 6, 4, "Pricing"),
    ("Top Cuisine
Promotion", 5, 2, "Revenue"),
    ("Double Acquisition
Channel spend", 6, 3, "Customer"),
    ("Low-Rating Brand
Audit", 7, 5, "Operations"),
    ("Payment Retry
Flow", 5, 3, "Operations"),
    ("Loyalty Program
Launch", 8, 8, "Customer"),
]

labels = [a[0] for a in actions]
impact = [a[1] for a in actions]
effort = [a[2] for a in actions]
cats   = [a[3] for a in actions]

cat_colors = {"Operations":"#E63946","Customer":"#2A9D8F",
              "Revenue":"#E9C46A","Pricing":"#457B9D"}
colors_a = [cat_colors[c] for c in cats]

fig, ax = plt.subplots(figsize=(11, 8))
scatter = ax.scatter(effort, impact, s=300, c=colors_a, alpha=0.85,
                     edgecolors="white", linewidths=1.5, zorder=3)

for lbl, eff, imp in zip(labels, effort, impact):
    ax.annotate(lbl, (eff, imp),
                textcoords="offset points", xytext=(8, 4),
                fontsize=8.5, color="#333")

# Quadrant lines
ax.axhline(5.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axvline(5.5, color="grey", linestyle="--", linewidth=0.8, alpha=0.5)

# Quadrant labels
ax.text(1, 9.5, "QUICK WINS
(Do First)", fontsize=10, color="#43AA8B",
        fontweight="bold", alpha=0.7)
ax.text(7, 9.5, "BIG BETS
(Plan & Invest)", fontsize=10, color="#E9C46A",
        fontweight="bold", alpha=0.7)
ax.text(1, 1.2, "FILL-INS
(Nice to Have)", fontsize=10, color="#888",
        fontweight="bold", alpha=0.7)
ax.text(7, 1.2, "RECONSIDER
(Low ROI)", fontsize=10, color="#E63946",
        fontweight="bold", alpha=0.7)

# Legend
from matplotlib.patches import Patch
legend_els = [Patch(facecolor=v, label=k) for k,v in cat_colors.items()]
ax.legend(handles=legend_els, loc="lower right", title="Category")

ax.set_xlabel("Effort Required (1=Low, 10=High)", fontsize=11)
ax.set_ylabel("Business Impact (1=Low, 10=High)", fontsize=11)
ax.set_title("Action Priority Matrix — Impact vs Effort",
             fontsize=14, fontweight="bold")
ax.set_xlim(0, 11); ax.set_ylim(0, 11)
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()


## 8. KPI Targets & Success Metrics

In [ ]:

kpi_targets = [
    # (KPI, Current, Target, Timeframe, Owner)
    ("Delivery Rate",        f"{delivery_rate:.1f}%",  "70%",   "6 months", "Operations"),
    ("Cancellation Rate",    f"{cancel_rate:.1f}%",    "<15%",  "6 months", "Operations"),
    ("Refund Rate",          f"{refund_rate:.1f}%",    "<10%",  "3 months", "Quality"),
    ("Avg Order Value",      f"₹{avg_order_value:,.0f}","₹950+", "4 months", "Product"),
    ("Champion Churn Rate",  "Unknown",                "0%",    "Ongoing",  "CRM"),
    ("At-Risk Reactivation", "0%",                     "15%",   "60 days",  "CRM"),
    ("MoM Revenue Growth",   f"{mom_growth:+.1f}%",   "+5%",   "Monthly",  "All"),
    ("New Customers/Month",  "~45",                    "55+",   "3 months", "Marketing"),
    ("Top Brand Refund Rate",f"{brand_refund_s.max():.1f}%","<15%","3 months","Partner Ops"),
    ("Discount % of GMV",    f"{total_discounts/gmv*100:.1f}%","<3.5%","6 months","Finance"),
]

print(f"{'KPI':<28} {'CURRENT':>10}  {'TARGET':>8}  {'TIMELINE':>10}  {'OWNER'}")
print("="*75)
for row in kpi_targets:
    status = "⚠️ " if "%" in row[1] and float(row[1].replace("%","").replace("₹","").replace("+","").replace(",","")) >              float(row[2].replace("%","").replace("<","").replace("+","").replace("₹","").replace(",",""))              else "✅ "
    try:
        print(f"  {row[0]:<26} {row[1]:>10}  {row[2]:>8}  {row[3]:>10}  {row[4]}")
    except:
        print(f"  {row[0]:<26} {row[1]:>10}  {row[2]:>8}  {row[3]:>10}  {row[4]}")
print("="*75)


In [ ]:

# Final visual summary — current vs target
metrics_v = ["Delivery Rate","Cancel Rate","Refund Rate"]
current_v  = [delivery_rate, cancel_rate, refund_rate]
target_v   = [70, 15, 10]
colors_ok  = ["#43AA8B" if (i==0 and c>=t) or (i>0 and c<=t) else "#E63946"
              for i,(c,t) in enumerate(zip(current_v, target_v))]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, metric, curr, tgt, col in zip(axes, metrics_v, current_v, target_v, colors_ok):
    ax.bar(["Current","Target"], [curr, tgt], color=[col, "#2A9D8F"], edgecolor="white", width=0.5)
    ax.set_title(metric, fontweight="bold")
    ax.set_ylabel("%")
    for i, v in enumerate([curr, tgt]):
        ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=12)
    gap = abs(curr - tgt)
    direction = "▲ Need +" if (metric=="Delivery Rate" and curr<tgt) else "▼ Need -"
    ax.set_xlabel(f"{direction}{gap:.1f}pp", fontsize=11, color=col)

plt.suptitle("KPI Targets: Current vs Goal", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("
🎯 Priority Actions This Week:")
print("  1. Launch cancel-reduction ETA notification (Quick Win)")
print("  2. Set refund-rate SLA and notify all restaurant partners (Quick Win)")
print("  3. Segment customers into RFM tiers in CRM (Quick Win)")
print("  4. Brief product team on AOV upsell feature (Quick Win)")
print("  5. Schedule loyalty program scoping with stakeholders (Big Bet)")


## ✅ Final Recommendations Summary

### 🔴 Do This Week (Quick Wins — High Impact, Low Effort)
1. **Protect Champions** — activate VIP perks immediately for top-value customers
2. **Win-back At-Risk segment** — automated email + push sequence
3. **Refund SLA** — notify all restaurant partners of the 15% refund threshold
4. **Top Cuisine Promotion** — feature on homepage and push notification

### 🟠 Do This Month (High Impact, Medium Effort)
5. **ETA & cancellation flow** — real-time restaurant acceptance notification
6. **AOV upsell nudges** — add "Frequently ordered together" at checkout in low-AOV cities
7. **Personalised discounts** — replace blanket discounts with segment-based offers
8. **Double acquisition spend** on the top-performing channel

### 🟢 Plan for Next Quarter (Big Bets)
9. **Loyalty Programme** — tiered rewards system (Bronze / Gold / Platinum)
10. **Expand top-city model** — replicate best city's operational playbook elsewhere
11. **Restaurant quality programme** — monthly audits for brands below 4.0★
12. **Payment retry flow** — reduce cancellations triggered by payment failure
